<a href="https://colab.research.google.com/github/open-mmlab/mmpose/blob/master/demo/MMPose_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MMPose Tutorial

Welcome to MMPose colab tutorial! In this tutorial, we will show you how to
- perform inference with an MMPose model
- train a new mmpose model with your own datasets

Let's start!

## Install MMPose

We recommend to use a conda environment to install mmpose and its dependencies. And compilers `nvcc` and `gcc` are required.

In [ ]:
# check NVCC version
!nvcc -V

# check GCC version
!gcc --version

# check python in conda environment
!which python

In [ ]:
# install dependencies: (use cu111 because colab has CUDA 11.1)
%pip install torch==1.10.0+cu111 torchvision==0.11.0+cu111 -f https://download.pytorch.org/whl/torch_stable.html

# install mmcv-full thus we could use CUDA operators
# %pip install mmcv-full -f https://download.openmmlab.com/mmcv/dist/cu111/torch1.10.0/index.html
%pip install mmengine

%pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu111/torch1.10.0/index.html
# install mmdet for inference demo
%pip install mmdet

# clone mmpose repo
%rm -rf mmpose
!git clone https://github.com/open-mmlab/mmpose.git
%cd mmpose

# install mmpose dependencies
%pip install -r requirements.txt

# install mmpose in develop mode
%pip install -e .

In [ ]:
# Check Pytorch installation
import torch, torchvision

print('torch version:', torch.__version__, torch.cuda.is_available())
print('torchvision version:', torchvision.__version__)

# Check MMPose installation
import mmpose

print('mmpose version:', mmpose.__version__)

# Check mmcv installation
from mmcv.ops import get_compiling_cuda_version, get_compiler_version

print('cuda version:', get_compiling_cuda_version())
print('compiler information:', get_compiler_version())

## Inference with an MMPose model

MMPose provides high level APIs for model inference and training.

In [ ]:
import os
import tempfile
from IPython.display import Image, display
import cv2
from mmpose.apis import (inference_topdown, init_model, visualize)

from mmdet.apis import inference_detector, init_detector

import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

local_runtime = False

try:
    from google.colab.patches import cv2_imshow  # for image visualization in colab
except:
    local_runtime = True

In [ ]:
def process_mmdet_results(mmdet_results, cat_id=1, bbox_thr=None):
  """Process mmdet results, and return a list of bboxes.

  Args:
      mmdet_results (list|tuple): mmdet results.
      bbox_thr (float): threshold for bounding boxes.
      cat_id (int): category id (default: 1 for human)

  Returns:
      person_results (list): a list of detected bounding boxes
  """
  if isinstance(mmdet_results, tuple):
      det_results = mmdet_results[0]
  else:
      det_results = mmdet_results

  bboxes = det_results.pred_instances.bboxes.cpu()

  bboxes = np.array(bboxes)
  person_results = np.array([np.array(bbox, dtype=np.float32) for bbox in bboxes])

  if bbox_thr is not None:
      assert bboxes.shape[-1] == 5
      valid_idx = np.where(bboxes[:, 4] > bbox_thr)[0]
      bboxes = bboxes[valid_idx]

  """for bbox in bboxes:
      person = {}
      person['bbox'] = np.array(bbox, dtype=np.float32)
      person_results.append(person)"""

  return person_results

In [ ]:
# DataFrame to store angles
angles_columns = ['frame', 'left_elbow_angle', 'left_knee_angle', 'left_shoulder_angle', 'left_hip_angle', 'left_ankle_angle',
            'right_elbow_angle', 'right_knee_angle', 'right_shoulder_angle', 'right_hip_angle', 'right_ankle_angle']

joints_columns = ['frame', 'left_elbow', 'left_knee', 'left_shoulder', 'left_hip', 'left_ankle', 'left_wrist','left_feet','right_shoulder',
                   'right_elbow', 'right_wrist', 'right_hip', 'right_knee', 'right_ankle','right_feet']


mm_joints_df = pd.DataFrame(columns=joints_columns)

mm_angles_df = pd.DataFrame(columns=angles_columns)

In [ ]:
def calculate_angle(a, b, c):
    a = np.array(a)  # First point
    b = np.array(b)  # Mid point
    c = np.array(c)  # End point

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360.0 - angle

    return angle

In [ ]:


#pose_config = 'configs/wholebody_2d_keypoint/rtmpose/coco-wholebody/rtmpose-l_8xb32-270e_coco-wholebody-384x288.py'
#pose_checkpoint = 'configs/wholebody_2d_keypoint/rtmpose/coco-wholebody/rtmpose-l_simcc-ucoco_dw-ucoco_270e-384x288-2438fd99_20230728.pth'
pose_config = 'configs/body_2d_keypoint/rtmpose/body8/rtmpose-x_8xb256-700e_body8-halpe26-384x288.py'
pose_checkpoint = 'configs/body_2d_keypoint/rtmpose/body8/rtmpose-x_simcc-body7_pt-body7-halpe26_700e-384x288-7fb6e239_20230606.pth'
det_config = 'demo/mmdetection_cfg/faster_rcnn_r50_fpn_coco.py'
det_checkpoint = 'https://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_1x_coco/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth'
# Initialize models
pose_model = init_model(pose_config, pose_checkpoint)
det_model = init_detector(det_config, det_checkpoint)

# Directory containing images
#image_dir = '../maxV'  # Update with the actual directory path
#image_list = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith('.jpg')]

# Directory to save results
#output_dir = '../pose_results'
#os.makedirs(output_dir, exist_ok=True)


In [ ]:


# Keypoint labels
keypoint_labels = {
    0: "Nose", 1: "LEye", 2: "REye", 3: "LEar", 4: "REar", 5: "LShoulder", 6: "RShoulder",
    7: "LElbow", 8: "RElbow", 9: "LWrist", 10: "RWrist", 11: "LHip", 12: "RHip",
    13: "LKnee", 14: "Rknee", 15: "LAnkle", 16: "RAnkle", 17: "Head", 18: "Neck",
    19: "Hip", 20: "LBigToe", 21: "RBigToe", 22: "LSmallToe", 23: "RSmallToe",
    24: "LHeel", 25: "RHeel"
}

keypoints_data = {}

for img in image_list:

    pose_model = init_model(pose_config, pose_checkpoint)
    det_model = init_detector(det_config, det_checkpoint)
    # Inference detection
    mmdet_results = inference_detector(det_model, img)

    # Extract person bounding boxes
    person_results = process_mmdet_results(mmdet_results, cat_id=1)

    if not person_results.any():
        print(f"No person detected in {img}")
        continue

    # Inference pose
    pose_results = inference_topdown(pose_model, img, person_results)


    # Extract keypoints correctly
    keypoints = pose_results[0].pred_instances.keypoints
    keypoints_data[img] = {keypoint_labels[i]: keypoints[0][i].tolist() for i in range(len(keypoints[0]))}


    # Visualize results
    vis_result = visualize(img, keypoints)
    vis_result = cv2.resize(vis_result, dsize=None, fx=0.5, fy=0.5)

    # Save visualization
    output_path = os.path.join(output_dir, os.path.basename(img).replace('.jpg', '_pose.jpg'))
    cv2.imwrite(output_path, vis_result)

    # Display results if running locally
    if local_runtime:
        with tempfile.TemporaryDirectory() as tmpdir:
            file_name = os.path.join(tmpdir, 'pose_results.png')
            cv2.imwrite(file_name, vis_result)
            display(Image(file_name))
    else:
        cv2_imshow(vis_result)

# Save keypoints data
keypoints_file = os.path.join(output_dir, 'keypoints.json')
with open(keypoints_file, 'w') as f:
    json.dump(keypoints_data, f, indent=4)

print(f"Keypoints saved in {keypoints_file}")
print(f"Pose images saved in {output_dir}")



In [ ]:

# Function to process a single video using YOLO and calculate joint angles
def process_single_video(video_file_path, frame_numbers, mm_angles_df, mm_joints_df):
    cap = cv2.VideoCapture(video_file_path)

    if not cap.isOpened():
        print(f"Error opening video file {video_file_path}")
        return

    print(f"Processing video: {video_file_path}")
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        pose_model = init_model(pose_config, pose_checkpoint)
        det_model = init_detector(det_config, det_checkpoint)

        mmdet_results = inference_detector(det_model, frame)

        # Extract person bounding boxes
        person_results = process_mmdet_results(mmdet_results, cat_id=1)

        if not person_results.any():
            print(f"No person detected in {frame}")
            continue

        # Inference pose
        pose_results = inference_topdown(pose_model, frame, person_results)


        # Extract keypoints correctly
        keypoints = pose_results[0].pred_instances.keypoints[0]

        for keypoint in keypoints:
            keypoint = keypoint.tolist()
            cv2.circle(frame, (int(keypoint[0]), int(keypoint[1])), 5, (0, 255, 0), -1)

        left_shoulder = -1
        left_elbow = -1
        left_wrist = -1
        left_hip = -1
        left_knee = -1
        left_ankle = -1

        # Get coordinates for right side
        right_shoulder = -1
        right_elbow = -1
        right_wrist = -1
        right_hip = -1
        right_knee = -1
        right_ankle = -1

        # # Calculate angles for left side
        left_elbow_angle = -1
        left_knee_angle = -1
        left_shoulder_angle = -1
        left_hip_angle = -1
        # left_ankle_angle = calculate_angle(left_knee, left_ankle, [left_ankle[0], left_ankle[1] + 0.1])  # Assuming vertical line for ankle

        # # Calculate angles for right side
        right_elbow_angle = -1
        right_knee_angle = -1
        right_shoulder_angle = -1
        right_hip_angle = -1

        # Extract landmarks
        if len(keypoints) > 0:

            """
            0: "Nose", 1: "LEye", 2: "REye", 3: "LEar", 4: "REar", 5: "LShoulder", 6: "RShoulder",
            7: "LElbow", 8: "RElbow", 9: "LWrist", 10: "RWrist", 11: "LHip", 12: "RHip",
            13: "LKnee", 14: "Rknee", 15: "LAnkle", 16: "RAnkle", 17: "Head", 18: "Neck",
            19: "Hip", 20: "LBigToe", 21: "RBigToe", 22: "LSmallToe", 23: "RSmallToe",
            24: "LHeel", 25: "RHeel"
            """
            # Get coordinates for left side
            left_shoulder = keypoints[5]
            left_elbow = keypoints[7]
            left_wrist = keypoints[9]
            left_hip = keypoints[11]
            left_knee = keypoints[13]
            left_ankle = keypoints[15]
            left_feet = keypoints[20]

            # Get coordinates for right side
            right_shoulder = keypoints[6]
            right_elbow = keypoints[8]
            right_wrist = keypoints[10]
            right_hip = keypoints[12]
            right_knee = keypoints[14]
            right_ankle = keypoints[16]
            right_feet = keypoints[21]

            # # Calculate angles for left side
            left_elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
            left_knee_angle = calculate_angle(left_hip, left_knee, left_ankle)
            left_shoulder_angle = calculate_angle(left_hip, left_shoulder, left_elbow)
            left_hip_angle = calculate_angle(left_shoulder, left_hip, left_knee)
            left_ankle_angle = calculate_angle(left_knee, left_ankle, left_feet)

            # # Calculate angles for right side
            right_elbow_angle = calculate_angle(right_shoulder, right_elbow, right_wrist)
            right_knee_angle = calculate_angle(right_hip, right_knee, right_ankle)
            right_shoulder_angle = calculate_angle(right_hip, right_shoulder, right_elbow)
            right_hip_angle = calculate_angle(right_shoulder, right_hip, right_knee)
            right_ankle_angle = calculate_angle(right_knee, right_ankle, right_feet)

            for point in [left_shoulder, left_elbow, left_wrist, left_hip, left_knee, left_ankle, left_feet,
                        right_shoulder, right_elbow, right_wrist, right_hip, right_knee, right_ankle, right_feet]:
                cv2.circle(frame, tuple(np.multiply(point, [1, 1]).astype(int)), 5, (0, 0, 255), -1)
            # Annotate angles on the image
            cv2.putText(frame, f'Left Elbow: {int(left_elbow_angle)}', tuple(np.multiply(left_elbow, [1, 1]).astype(int)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Left Knee: {int(left_knee_angle)}', tuple(np.multiply(left_knee, [1, 1]).astype(int)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Left Shoulder: {int(left_shoulder_angle)}', tuple(np.multiply(left_shoulder, [1, 1]).astype(int)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Left Hip: {int(left_hip_angle)}', tuple(np.multiply(left_hip, [1, 1]).astype(int)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Left Ankle: {int(left_ankle_angle)}', tuple(np.multiply(left_ankle, [1, 1]).astype(int)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)

            cv2.putText(frame, f'Right Elbow: {int(right_elbow_angle)}', tuple(np.multiply(right_elbow, [1, 1]).astype(int)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Right Knee: {int(right_knee_angle)}', tuple(np.multiply(right_knee, [1, 1]).astype(int)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Right Shoulder: {int(right_shoulder_angle)}', tuple(np.multiply(right_shoulder, [1, 1]).astype(int)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Right Hip: {int(right_hip_angle)}', tuple(np.multiply(right_hip, [1, 1]).astype(int)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Right Ankle: {int(right_ankle_angle)}', tuple(np.multiply(right_ankle, [1, 1]).astype(int)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)

        if frame_count in frame_numbers:
                # Get the name of the video
                videoname = video_file_path.split('/')[-1].split('.')[0]
                outdir = "../mmpose_halpe_"+videoname
                # Save the frame with annotated points
                output_frame_path = os.path.join(outdir, f"frame_{frame_count}_pose.jpg")
                cv2.imwrite(output_frame_path, frame)

                # Store points into a DataFrame
                mm_joints_df = pd.concat([mm_joints_df, pd.DataFrame([{
                    'frame': f"frame_{frame_count}",
                    'left_shoulder': left_shoulder,
                    'left_elbow': left_elbow,
                    'left_wrist': left_wrist,
                    'left_hip': left_hip,
                    'left_knee': left_knee,
                    'left_ankle': left_ankle,
                    'left_feet': left_feet,
                    'right_shoulder': right_shoulder,
                    'right_elbow': right_elbow,
                    'right_wrist': right_wrist,
                    'right_hip': right_hip,
                    'right_knee': right_knee,
                    'right_ankle': right_ankle,
                    'right_feet': right_feet
                }])], ignore_index=True)

                mm_angles_df = pd.concat([mm_angles_df, pd.DataFrame([{
                    'frame': f"frame_{frame_count}",
                    'left_elbow_angle': left_elbow_angle,
                    'left_knee_angle': left_knee_angle,
                    'left_shoulder_angle': left_shoulder_angle,
                    'left_hip_angle': left_hip_angle,
                    'left_ankle_angle': left_ankle_angle,
                    'right_elbow_angle': right_elbow_angle,
                    'right_knee_angle': right_knee_angle,
                    'right_shoulder_angle': right_shoulder_angle,
                    'right_hip_angle': right_hip_angle,
                    'right_ankle_angle': right_ankle_angle
                }])], ignore_index=True)

                mm_joints_df.to_json(os.path.join(outdir, 'keypoints.json'), orient='records')
                mm_angles_df.to_json(os.path.join(outdir, 'angles.json'), orient='records')

        frame_count += 1

        # Display the frame
        cv2_imshow(frame)

        # Press Q on keyboard to exit
        if cv2.waitKey(25) & 0xFF == ord('q'):
            break

    # Release the video capture object
    cap.release()
    cv2.destroyAllWindows()

    return mm_angles_df

# Function to process videos using YOLO and calculate joint angles
def process_videos_with_mm(formatted_side_path):
    # Iterate over files in the formatted side directory
    for filename in os.listdir(formatted_side_path):
        if filename.lower().endswith('.mov'):
            base_filename = filename.split('.')[0]
            # Get the list of all frame files in the folder
            frame_folder_path = os.path.join(formatted_side_path, base_filename)
            frame_files = [f for f in os.listdir(frame_folder_path) if f.startswith('frame_') and f.endswith('.jpg')]

            # Extract frame numbers from the filenames and sort them
            frame_numbers = sorted([int(f.split('_')[1].split('.')[0]) for f in frame_files])
            video_file_path = os.path.join(formatted_side_path, filename)
            process_single_video(video_file_path, frame_numbers, mm_angles_df, mm_joints_df)


process_videos_with_mm("../videos/")